# Instructions

The places where you have enter code, to answer the questions, are marked with `# YOUR CODE HERE`. Once you have written your code you should remove the `raise NotImplementedError()` statement.

The first two questions are on phase estimation and the rest is about order finding algorithm.

## Question 1 (2 points)

Complete the function `create_operator` that returns a controlled $U$ gate where $U$ is an operator with eigenvector $|1\rangle$ and eigenvalue $e^{2\pi i \phi}$.

The function `create_operator` has

- Input: $\phi$
- Returns: Controlled $U$ gate with the properties described above

Don't create any circuit. Only a gate should be returned.

In [4]:
%pip install cirq

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 1.2 MB/s eta 0:00:02
   --------- ------------------------------ 0.5/2.1 MB 1.2 MB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.1 MB 909.4 kB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.1 MB 909.4 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 715.9 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 715.9 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 715.9 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 715.9 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 715.9 kB/s eta 0:00:02
   ------------------------ --------------- 1.3/2.1 MB 536.1 kB/s eta 0:00:02
   -----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 6.1.0 requires ipython!=8.17.1,!=9.1.0,!=9.2.0,!=9.3.0,!=9.4.0,<10.0.0,>=8.15.0, but you have ipython 9.4.0 which is incompatible.


In [5]:
from cirq import CZPowGate

def create_operator(phi):
    # YOUR CODE HERE
    CU = CZPowGate(exponent=2*phi)
    
    # Do not modify anything below this line  
    return CU

In [6]:
# You can use this code to test your function by checking the bottom right corner of the unitary matrix
import cirq
from cmath import exp, pi

def test(phi):
    CU = create_operator(phi)
    unitary_matrix = cirq.unitary(CU)
    print(unitary_matrix)

phi=3/5
test(phi)
print(exp(2*pi*phi*1j))

[[ 1.        +0.j          0.        +0.j          0.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j          1.        +0.j          0.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          1.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          0.        +0.j
  -0.80901699-0.58778525j]]
(-0.8090169943749475-0.587785252292473j)


In [7]:
# hidden tests in this cell will be used for grading.

In [8]:
# hidden tests in this cell will be used for grading.

## Question 2 (8 points)

Complete the function `estimate_phi` such that given a controlled $U$ gate where $U$ is an operator with eigenvector $|1\rangle$ and eigenvalue $e^{2\pi i \phi}$, it estimates and returns $\phi$.

The function `estimate_phi` has

- Input: Controlled $U$ gate
- Returns: Estimate for phi 

You are given iqft and qpe algorithms and you can use them in your solution. Let the size of the first register equal 10.

In [9]:
def iqft(n,qubits,circuit):
    
    #Swap the qubits
    for i in range(n//2):
        circuit.append(SWAP(qubits[i],qubits[n-i-1]), strategy = InsertStrategy.NEW)
     
    #For each qubit
    for i in range(n-1,-1,-1):
        #Apply CR_k gates where j is the control and i is the target
        k=n-i #We start with k=n-i
        for j in range(n-1,i,-1):
            #Define and apply CR_k gate
            crk = CZPowGate(exponent = -2/2**(k))
            circuit.append(crk(qubits[j],qubits[i]),strategy = InsertStrategy.NEW)
            k=k-1 #Decrement at each step
            
        #Apply Hadamard to the qubit
        circuit.append(H(qubits[i]),strategy = InsertStrategy.NEW)

def qpe(t,control, target, circuit, CU):
    
    #Apply Hadamard to control qubits
    circuit.append(cirq.H.on_each(control))
    
    #Apply CU gates
    for i in range(t):
        #Obtain the power of CU gate 
        CUi = CU**(2**i)
        #Apply CUi gate where t-i-1 is the control
        circuit.append(CUi(control[t-i-1],*target))
        
    #Apply inverse QFT
    iqft(t,control,circuit)

In [10]:
import cirq
from cirq.circuits import InsertStrategy
from cirq import H, SWAP, CZPowGate
def estimate_phi(mystery):
    

    #Create cirucit
    circuit = cirq.Circuit()
    t = 10
    # Crear registros de control y objetivo
    control = [cirq.LineQubit(i) for i in range(t)]
    target = [cirq.LineQubit(t)]
    
    # Inicializar el target en el estado |1> aplicando una compuerta X
    circuit.append(cirq.X(*target))
    
    # Aplicar QPE
    qpe(t, control, target, circuit, mystery)
    
    # Medir el registro de control
    circuit.append(cirq.measure(*control, key='result'))
    s=cirq.Simulator()
    samples=s.run(circuit, repetitions=1000)
    
    #Most frequent observation
    freq = list(samples.histogram(key='result').keys())[0]
    
    return freq/2**t

In [11]:
import math
# You can use this code to test your function by different operators
def test_qpe(phi):
    operator = CZPowGate(exponent=2*phi)
    return estimate_phi(operator)

assert(math.isclose(test_qpe(0.23),0.23,rel_tol=1e-2))

In [12]:
# hidden tests will be used for grading.

## Question 3 (1 point)

Complete the function `calcL` such that given $N$, it returns the size $L$ of the target register. Note that $L = \log_2 N$. 

The function `calcL` has

- Input: $N$
- Returns: size of the target register

In [13]:
import math
def calcL(N):
    L = math.ceil(math.log2(N))
    return L

In [14]:
# You can use this cell to test your function

print(calcL(30))
print(calcL(62))

5
6


In [15]:
# hidden tests will be used for grading.

## Question 4 (1 point)

Complete the function `calct` such that given $N$, it returns the size $t$ of the control register. Note that $t = 2L + 1 + \big \lceil \log \big( 2 + \frac{1}{2\epsilon} \big) \big \rceil$. Let $\epsilon=0.1$. 

The function `calct` has

- Input: $N$
- Returns: size of the control register

In [16]:
import math
def calct(N):
    L = math.ceil(math.log2(N))
    epsilon = 0.1
    t = 2 * L + 1 + math.ceil(math.log2(2 + 1 / (2 * epsilon)))
    return t

In [17]:
# You can use this cell to test your function

print(calct(30)) #For 30 the answer is 14
print(calct(62)) #For 62 the answer is 16

14
16


In [18]:
# hidden tests will be used for grading.

## Question 5 ( 2 points)

Complete the function `create_regs` such that given the size of the control and target registers, it returns the control and target registers.

The function `create_regs` has

- Input: $L$ and $t$
- Returns: Two registers `control` and `target`

In [19]:
import cirq
def create_regs(t,L):

    control = [cirq.LineQubit(i) for i in range(t)]
    target = [cirq.LineQubit(i + t) for i in range(L)]
    return control, target

In [20]:
# hidden tests will be used for grading.
# If this cell results in an error, your implementation is incorrect

In [21]:
# hidden tests will be used for grading.
# If this cell results in an error, your implementation is incorrect

## Question 6 (8 points)

Complete the function `order` so that given $x$ and $N$, it returns the histogram of samples from the order finding circuit.

The function `order` has

- Input: $x$ and $N$
- Returns: A histogram of samples

Notes:

- You are given a function named $U_x$ which implements $U_x |y\rangle \rightarrow |xy {\mod N}\rangle $ and returns its controlled version. Run the following cell to load the function. In order to use the function you should pass $x$ and $N$ as parameter.

    <pre>CU=Ux(x,N)</pre>

- You are also given the code for `qpe` and `iqft`.

In [22]:
import numpy as np
def Ux(x,N):

    k=1
    while(N>2**k):
        k=k+1
        
    u = np.zeros([2**k, 2**k], dtype = int) 

    for i in range(N):
        u[x*i%N][i]=1
    for i in range(N,2**k):
        u[i][i]=1
        

    XU = cirq.MatrixGate(u).controlled()
    return XU

In [23]:
import cirq
from cirq.circuits import InsertStrategy
from cirq import H, SWAP, CZPowGate
def iqft(n,qubits,circuit):
    
    #Swap the qubits
    for i in range(n//2):
        circuit.append(SWAP(qubits[i],qubits[n-i-1]), strategy = InsertStrategy.NEW)
     
    #For each qubit
    for i in range(n-1,-1,-1):
        #Apply CR_k gates where j is the control and i is the target
        k=n-i #We start with k=n-i
        for j in range(n-1,i,-1):
            #Define and apply CR_k gate
            crk = CZPowGate(exponent = -2/2**(k))
            circuit.append(crk(qubits[j],qubits[i]),strategy = InsertStrategy.NEW)
            k=k-1 #Decrement at each step
            
        #Apply Hadamard to the qubit
        circuit.append(H(qubits[i]),strategy = InsertStrategy.NEW)

def qpe(t,control, target, circuit, CU):
    
    #Apply Hadamard to control qubits
    circuit.append(cirq.H.on_each(control))
    
    #Apply CU gates
    for i in range(t):
        #Obtain the power of CU gate 
        CUi = CU**(2**i)
        #Apply CUi gate where t-i-1 is the control
        circuit.append(CUi(control[t-i-1],*target))
        
    #Apply inverse QFT
    iqft(t,control,circuit)

In [24]:
import cirq
from cirq import X, SWAP

def order(x,N):
    #Create a circuit
    circuit = cirq.Circuit()
    # 1. Calcular L y t
    L = math.ceil(math.log2(N))
    epsilon = 0.1
    t = 2 * L + 1 + math.ceil(math.log2(2 + 1 / (2 * epsilon)))
    
    # 2. Crear los registros de control y objetivo
    control = [cirq.LineQubit(i) for i in range(t)]
    target = [cirq.LineQubit(i + t) for i in range(L)]
    
    # 3. Inicializar el registro target en el estado entero |1> (aplicando X al último bit)
    circuit.append(X(target[-1]))
    
    # 4. Obtener el operador Ux controlado
    CU = Ux(x, N)
    
    # 5. Aplicar QPE
    qpe(t, control, target, circuit, CU)

    #Measure the control register
    circuit.append(cirq.measure(*control, key='result'))

    #Sample the circuit
    s=cirq.Simulator()
    samples=s.run(circuit, repetitions=1000)

    # Return a histogram of samples
    results= samples.histogram(key='result')
    return results

In [25]:
#You can use this cell to test your function

results = order(4,31)
print(results)

Counter({0: 189, 13107: 188, 3277: 176, 9830: 121, 6554: 106, 9831: 54, 6553: 51, 3276: 9, 6555: 9, 9832: 8, 9829: 8, 6552: 8, 13108: 7, 9828: 5, 13106: 5, 3280: 4, 9827: 4, 6551: 4, 3278: 4, 13109: 3, 13105: 3, 3279: 3, 9834: 3, 6556: 3, 9833: 2, 13113: 2, 3275: 2, 9846: 1, 9824: 1, 6546: 1, 6550: 1, 13260: 1, 13101: 1, 9825: 1, 6559: 1, 6523: 1, 13102: 1, 6549: 1, 9857: 1, 6548: 1, 6566: 1, 9814: 1, 9835: 1, 6560: 1, 13117: 1, 6561: 1})


In [26]:
# hidden tests will be used for grading.
# If this cell results in an error, your implementation is incorrect